# CIFAR 用の DeepInversion ノートブック

In [1]:
import os
import sys
import glob
import numpy as np
import json
import random
import collections


import torch
import torch.optim as optim
import torchvision.utils as vutils

import torch.nn as nn
import torch.nn.functional as F



## GPUの設定

In [2]:
# 使用するgpuを指定
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

## パスの設定

In [3]:
# ベース部分のパス
ckpt_path = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL/checkpoint"

# cifar100のbaseline用パス
base_cifar100_path = "baseline/cifar100"

# baseline
method = "baseline"
baseline_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0_0_0")

# baseline_mu
method = "baseline_mu"
baseline_mu_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0_0_1.0")
# print("baseline_mu_path: ", baseline_mu_path)

# prl2
method = "prl2"
prl_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0.1_2_1.0")
# print("prl_path: ", prl_path)

# prl_mu
method = "prl-mu"
prl_mu_path = os.path.join(ckpt_path, method, base_cifar100_path, "50/5/10_10_0.1_2_1.0")
# print("prl_mu_path: ", prl_mu_path)

## Jsonファイルの読み込みからモデルの用意

In [4]:
# 手法名（変更箇所）
method = "BASELINE"

# プロジェクト root を sys.path に追加
project_root = "/home/kouyou/ContinualLearning/repexp/NeurIPS2024-PRL"
sys.path.append(project_root)

from utils import factory
import models

# --- 1) 設定を記述した jsonファイル の内容を読む ---
with open(os.path.join(project_root, "exps", method, "cifar.json")) as f:
    args = json.load(f)


args["device"] = ["0"]            # 必要に応じて
# args["model_name"] = "baseline" 


# --- 2) learner とネットワークの作成 ---
learner = factory.get_model(args["model_name"], args)
net = learner._network


# --- 3) checkpoint の読み込み ---
ckpt_dir = baseline_path
ckpt_file = os.path.join(ckpt_dir, "phase0.pkl")       # 読み込むモデルの指定

ckpt = torch.load(ckpt_file, map_location="cuda:0")
state_dict = ckpt["model_state_dict"]
print(state_dict.keys())

# fc の出力次元を checkpoint から取得
num_outputs = state_dict["fc.weight"].shape[0]
print("num_outputs: ", num_outputs)

# fc層の出力次元数を変更
net.update_fc(num_outputs)

# state_dict の読み込み
net.load_state_dict(state_dict)

net.cuda().eval()

# 忘却クラスや class_order を取り出す
forget_classes = ckpt.get("forget_classes", None)
class_order = ckpt.get

<ipython-input-4-2cf33e4e3fda>:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_file, map_location="cuda:0")


odict_keys(['convnet.conv1.0.weight', 'convnet.conv1.1.weight', 'convnet.conv1.1.bias', 'convnet.conv1.1.running_mean', 'convnet.conv1.1.running_var', 'convnet.conv1.1.num_batches_tracked', 'convnet.layer1.0.conv1.weight', 'convnet.layer1.0.bn1.weight', 'convnet.layer1.0.bn1.bias', 'convnet.layer1.0.bn1.running_mean', 'convnet.layer1.0.bn1.running_var', 'convnet.layer1.0.bn1.num_batches_tracked', 'convnet.layer1.0.conv2.weight', 'convnet.layer1.0.bn2.weight', 'convnet.layer1.0.bn2.bias', 'convnet.layer1.0.bn2.running_mean', 'convnet.layer1.0.bn2.running_var', 'convnet.layer1.0.bn2.num_batches_tracked', 'convnet.layer1.1.conv1.weight', 'convnet.layer1.1.bn1.weight', 'convnet.layer1.1.bn1.bias', 'convnet.layer1.1.bn1.running_mean', 'convnet.layer1.1.bn1.running_var', 'convnet.layer1.1.bn1.num_batches_tracked', 'convnet.layer1.1.conv2.weight', 'convnet.layer1.1.bn2.weight', 'convnet.layer1.1.bn2.bias', 'convnet.layer1.1.bn2.running_mean', 'convnet.layer1.1.bn2.running_var', 'convnet.layer

## Hookの設定

In [5]:
class DeepInversionFeatureHook():
    '''
    Implementation of the forward hook to track feature statistics and compute a loss on them.
    Will compute mean and variance, and will use l2 as a loss
    '''

    def __init__(self, module):
        self.hook = module.register_forward_hook(self.hook_fn)


    def hook_fn(self, module, input, output):
        # hook co compute deepinversion's feature distribution regularization
        nch = input[0].shape[1]

        mean = input[0].mean([0, 2, 3])
        var = input[0].permute(1, 0, 2, 3).contiguous().view([nch, -1]).var(1, unbiased=False)

        # forcing mean and variance to match between two distributions
        # other ways might work better, e.g. KL divergence
        r_feature = torch.norm(module.running_var.data.type(var.type()) - var, 2) + torch.norm(
            module.running_mean.data.type(var.type()) - mean, 2)

        self.r_feature = r_feature
        # must have no output

    def close(self):
        self.hook.remove()


In [9]:
# 設定
args = {}

# args["bs"] = 250
# args["iters_mi"] = 4000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.05
# args["di_var_scale"] = 2.5e-5
# args["di_l2_scale"] = 0.0
# args["r_feature_weight"] = 10
# args["exp_descr"] = "debug_cifar100"
# args["size"] = 32

# args["bs"] = 250
# args["iters_mi"] = 2000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.1
# args["di_var_scale"] = 0.001
# args["di_l2_scale"] = 0.0
# args["r_feature_weight"] = 10
# args["exp_descr"] = "debug_cifar100_v2"
# args["size"] = 32

# args["bs"] = 250
# args["iters_mi"] = 2000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.1
# args["di_var_scale"] = 0.001
# args["di_l2_scale"] = 0.0
# args["r_feature_weight"] = 15
# args["exp_descr"] = "debug_cifar100_v3"
# args["size"] = 32

# args["bs"] = 250
# args["iters_mi"] = 2000
# args["cig_scale"] = 0.0
# args["di_lr"] = 0.01
# args["di_var_scale"] = 0.001
# args["di_l2_scale"] = 0.0
# args["r_feature_weight"] = 20
# args["exp_descr"] = "debug_cifar100_v4"
# args["size"] = 32

args["bs"] = 100
args["iters_mi"] = 4000
args["cig_scale"] = 0.0
args["di_lr"] = 0.05
args["di_var_scale"] = 2.5e-5
args["di_l2_scale"] = 0.0
args["r_feature_weight"] = 10
args["exp_descr"] = "debug_cifar100_v5"
args["size"] = 64

random_labels = False

# 損失関数
criterion = nn.CrossEntropyLoss()

# 入力
data_type = torch.float
inputs = torch.randn((args["bs"], 3, args["size"], args["size"]), requires_grad=True, device='cuda', dtype=data_type)

# 最適化手法
optimizer = optim.Adam([inputs], lr=args["di_lr"])

batch_idx = 0
prefix = "runs/data_generation/"+args["exp_descr"]+"/"

for create_folder in [prefix, prefix+"/best_images/"]:
    if not os.path.exists(create_folder):
        os.makedirs(create_folder)

global_iteration = 0

In [10]:
best_cost = 1e6

# 入力の初期化
inputs.data = torch.randn((args["bs"], 3, args["size"], args["size"]), requires_grad=True, device='cuda')

# Optimizer の初期化
optimizer.state = collections.defaultdict(dict)

# ラベルの作成
if random_labels:
    targets = torch.LongTensor([random.randint(0,9) for _ in range(args["bs"])]).to('cuda')
else:
    targets = torch.LongTensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9,
                                10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
                                20, 21, 22, 23, 24, 25, 26, 27, 28, 29,
                                30, 31, 32, 33, 34, 35, 36, 37, 38, 39,
                                40, 41, 42, 43, 44, 45, 46, 47, 48, 49] * 2).to('cuda')

## Create hooks for feature statistics catching
loss_r_feature_layers = []
for module in net.modules():
    if isinstance(module, nn.BatchNorm2d):
        loss_r_feature_layers.append(DeepInversionFeatureHook(module))

# setting up the range for jitter
lim_0, lim_1 = 2, 2


# 学習部分
for epoch in range(args["iters_mi"]):
    
    # apply random jitter offsets
    off1 = random.randint(-lim_0, lim_0)
    off2 = random.randint(-lim_1, lim_1)
    inputs_jit = torch.roll(inputs, shifts=(off1,off2), dims=(2,3))

    # foward with jit images
    optimizer.zero_grad()
    net.zero_grad()
    outputs = net(inputs_jit)
    logits_all = outputs["logits"]
    logits = logits_all[:, ::4] 
    loss = criterion(logits, targets)
    loss_target = loss.item()

    # apply total variation regularization
    diff1 = inputs_jit[:,:,:,:-1] - inputs_jit[:,:,:,1:]
    diff2 = inputs_jit[:,:,:-1,:] - inputs_jit[:,:,1:,:]
    diff3 = inputs_jit[:,:,1:,:-1] - inputs_jit[:,:,:-1,1:]
    diff4 = inputs_jit[:,:,:-1,:-1] - inputs_jit[:,:,1:,1:]
    loss_var = torch.norm(diff1) + torch.norm(diff2) + torch.norm(diff3) + torch.norm(diff4)
    loss = loss + args["di_var_scale"] * loss_var

    # R_feature loss
    loss_distr = sum([mod.r_feature for mod in loss_r_feature_layers])
    loss = loss + args["r_feature_weight"] * loss_distr # best for noise before BN

    # l2 loss
    loss = loss + args["di_l2_scale"] * torch.norm(inputs_jit, 2)


    if epoch % 10==0:
        print(f"It {epoch}\t Losses: total: {loss.item():3.3f},\ttarget: {loss_target:3.3f} \tR_feature_loss unscaled:\t {loss_distr.item():3.3f}")
        vutils.save_image(inputs.data.clone(),
                            './{}/output_{}.png'.format(prefix, epoch//10),
                            normalize=True, scale_each=True, nrow=10)

    if best_cost > loss.item():
        best_cost = loss.item()
        best_inputs = inputs.data
    

    # 最適化実行
    loss.backward()
    optimizer.step()



name_use = "best_images"
if prefix is not None:
    name_use = prefix + name_use
next_batch = len(glob.glob("./%s/*.png" % name_use)) // 1

vutils.save_image(best_inputs[:20].clone(),
                    './{}/output_{}.png'.format(name_use, next_batch),
                    normalize=True, scale_each = True, nrow=10)

It 0	 Losses: total: 2392.255,	target: 3.971 	R_feature_loss unscaled:	 238.813


It 10	 Losses: total: 1336.770,	target: 3.918 	R_feature_loss unscaled:	 133.271
It 20	 Losses: total: 960.513,	target: 3.925 	R_feature_loss unscaled:	 95.645
It 30	 Losses: total: 706.515,	target: 3.926 	R_feature_loss unscaled:	 70.246
It 40	 Losses: total: 538.179,	target: 3.905 	R_feature_loss unscaled:	 53.415
It 50	 Losses: total: 452.540,	target: 3.886 	R_feature_loss unscaled:	 44.853
It 60	 Losses: total: 404.041,	target: 3.876 	R_feature_loss unscaled:	 40.004
It 70	 Losses: total: 368.904,	target: 3.869 	R_feature_loss unscaled:	 36.492
It 80	 Losses: total: 343.851,	target: 3.866 	R_feature_loss unscaled:	 33.987
It 90	 Losses: total: 359.568,	target: 3.856 	R_feature_loss unscaled:	 35.559
It 100	 Losses: total: 362.153,	target: 3.840 	R_feature_loss unscaled:	 35.819
It 110	 Losses: total: 327.846,	target: 3.846 	R_feature_loss unscaled:	 32.388
It 120	 Losses: total: 322.177,	target: 3.821 	R_feature_loss unscaled:	 31.824
It 130	 Losses: total: 356.660,	target: 3.840 	